# Fine-tuning open models with agents: from eval to deployment

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/langgraph-grafana-agent/langgraph-grafana-agent-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Durable, self-healing agents on Union, with end-to-end observability in Grafana.*

There are two agents in this notebook.

The **support agent** is the one in production. A ticket comes in, it routes it to a queue, it drafts a reply. Today its routing step is an API call per ticket: accurate, slow, billed. We're going to replace that call with a model we own, and we are not going to pick the model ourselves.

The **ML engineer** is a LangGraph agent that gets this request and operates the *model factory* on Union to fill it:

> Replace the support agent's router with a self-hosted open-source model: at least 95% accuracy on our tickets, under 150 ms per ticket on a T4. Baseline up to 5 candidates, fine-tune up to 3 of them, spend at most 12 runs.

| Step | You will | Minutes |
|---|---|---|
| 0 | Run the support agent as it is today. The "before". | 3 |
| 1 | Drive the factory yourself, no agent: parallel GPU evals, one fine-tune. | 5 |
| 2 | Hand the request to the ML engineer. It builds, deploys and tests the router. | 5 |
| 3 | Switch the support agent to the new router. The "after". | 3 |
| 4 | Look at both agents in Grafana. | 5 |
| 5 | Kill the engineer mid-run and watch it resume. | 5 |
| 6 | Bake off the engineer's brain (optional). | 5 |
| 7 | Drift: a new kind of ticket arrives. See it. | 3 |
| 8 | The loop closes itself: publish the tickets, a trigger notices, the engineer retrains. | 10 |

Every step is a Flyte task. `run(task)` sends it to the cluster you were given, prints the run URL, waits, and returns the result. **Open every run URL**: the run graph, the reports, and the Grafana links are the point.

---

## Setup

Run the next cell once. On Colab it clones the repo and installs the dependencies (a minute or two).

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone -q https://github.com/unionai/workshops
    %cd workshops/tutorials/langgraph-grafana-agent
    !pip install -q uv
    !uv pip install --system -q -r requirements.txt
    !uv pip install --system -q keyrings.alt pygments
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

import os
from getpass import getpass
from utils.file_viewer import view_file
from utils.workshop import run, show

### Your name on the cluster

Everyone in the room shares one project. Your tag namespaces the things that are yours: your serving app, your production artifacts, your triggers. Lowercase letters, digits, dashes.

In [ ]:
TAG = "yourname"                    # <-- change this
os.environ["FACTORY_TAG"] = TAG
os.environ["FLYTE_SECRET_PREFIX"] = TAG.upper().replace("-", "_") + "_"
print("your app and artifacts will be called ticket-router-" + TAG)

### Keys

The agents need a model. Claude is the default; to use OpenAI instead, set `AGENT_MODEL` to `openai:gpt-4.1` and provide `OPENAI_API_KEY`.

In [ ]:
os.environ["AGENT_MODEL"] = "anthropic:claude-opus-5"      # or "openai:gpt-4.1"
PROVIDER, KEY_VAR = ("openai", "OPENAI_API_KEY") if os.environ["AGENT_MODEL"].startswith("openai") else ("anthropic", "ANTHROPIC_API_KEY")
os.environ["FACTORY_PROVIDERS"] = PROVIDER
if not os.environ.get(KEY_VAR):
    os.environ[KEY_VAR] = getpass(KEY_VAR + ": ")

### Connect to the cluster

You were given an endpoint. The first call opens a login link; paste the code back here. Then put your key on the cluster as a secret (its name carries your tag, so it is yours).

In [ ]:
ENDPOINT = "demo.hosted.unionai.cloud"     # <-- the one you were given
!flyte create config --endpoint {ENDPOINT} --project flytesnacks --domain development --builder remote
!flyte create secret {os.environ["FLYTE_SECRET_PREFIX"]}{KEY_VAR} --value {os.environ[KEY_VAR]}

### Grafana (optional)

If you have a Grafana Cloud stack with Agent Observability enabled, put its values in the environment and every run from here on exports to it. If not, skip this: everything else works, and the host will show theirs.

In [ ]:
# os.environ["GRAFANA_HOST"] = "https://<stack>.grafana.net"
# os.environ["AGENTO11Y_ENDPOINT"] = "https://agento11y-prod-<region>.grafana.net"     # Agent Observability -> Configuration
# os.environ["AGENTO11Y_AUTH_TENANT_ID"] = "<instance id>"
# os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://otlp-gateway-prod-<region>.grafana.net/otlp"
# os.environ["GRAFANA_TOKEN"] = getpass("GRAFANA_TOKEN (glc_...): ")
# !flyte create secret {os.environ["FLYTE_SECRET_PREFIX"]}GRAFANA_TOKEN --value {os.environ["GRAFANA_TOKEN"]}

---

## 0. The support agent, as it is today

Thirty held-out tickets through the support agent: routed to a queue by the API model, then a two-line draft reply. Open the run and look at the report: routing accuracy, p50 route latency, tokens, and cost per 1,000 tickets. This is the "before".

In [ ]:
from support_agent import handle_tickets
before = run(handle_tickets, n=30, router="llm")
show(before, ["routing_accuracy", "route_p50_ms", "cost_per_1000_tickets_usd", "tokens"])

The agent itself is two LangGraph nodes, `route → draft`, each a durable step. `--router llm` is today; `--router oss` is the app we are about to build.

In [ ]:
view_file("support_agent.py", show_path=True)

---

## 1. The factory, with you at the controls

Before the engineer touches anything, drive the factory yourself: baseline three candidates (three T4 containers at once), fine-tune one, recheck it. Open the run: the evals are side by side, each with its own report, and the fine-tune report draws its loss curve live.

These results are cached. When the engineer asks for the same eval in step 2, it gets the answer in a second, and so does everyone else in the room.

In [ ]:
from step1_factory import model_factory
run(model_factory, fine_tune_too=True)

### The candidates and the numbers

| candidate | zero-shot | after fine-tune | p50 on T4 |
|---|---|---|---|
| smollm2-360m | 13% | | 110 ms |
| qwen2.5-0.5b | 59% | 1 epoch: 97.5% | 68 ms |
| qwen2.5-1.5b | 79% | 1 epoch: 97.5% | 83 ms |
| smollm2-1.7b | 36% | | 72 ms |
| modernbert-base (an encoder, not a chat model) | 9%, untrained head | 3 epochs, 16 s: 99.2% | 14 ms |

Nothing passes zero-shot. The bar is 95%. The engineer has to work that out from numbers it produces itself.

In [ ]:
view_file("tools.py", show_path=True)

---

## 2. Hand the request to the ML engineer

The engineer's graph is `think → tools → think … → decision`. Two nodes come from Flyte's LangGraph plugin (every model turn is recorded for replay; every tool call is a durable child action); the parallel tool node and the typed decision are ours.

Open the run while it works. Every action is named after what it does (`run_eval · qwen2.5-0.5b`, `fine_tune · modernbert-base · 3ep`), evals fan out across T4s, and at the end `promote` publishes your `ticket-router-<tag>` artifact and deploys your app, and `test_deployment` calls it. The parent report is the decision page: what was promoted, pass/fail against the request, everything it measured, the sequence of calls, the rationale.

In [ ]:
from step2_engineer import engineer
decision = run(engineer)
show(decision["decision"], ["action", "promoted_model", "accuracy", "latency_p50_ms", "deployment_test_passed", "runs_spent"])

In [ ]:
view_file("graph.py", show_path=True)

---

## 3. Switch the support agent to the new router

Same thirty tickets, same draft replies, but routing is now an HTTP call to your app. Put this report next to step 0's: accuracy holds, latency drops, and the cost that remains is the reply drafts.

In [ ]:
after = run(handle_tickets, n=30, router="oss")
show({"before": {k: before[k] for k in ("routing_accuracy", "route_p50_ms", "cost_per_1000_tickets_usd")},
      "after":  {k: after[k]  for k in ("routing_accuracy", "route_p50_ms", "cost_per_1000_tickets_usd")}})

---

## 4. Both agents in Grafana

Nothing to run. Open the step 2 run in the Flyte UI: the task has two links, **Grafana Agent Observability** (this run as a conversation: every generation with prompt, answer, tokens and cost; every tool call as a step) and **Grafana trace** (its spans in Tempo, where a `fine_tune` span is a minute wide). The support agent's runs from steps 0 and 3 are conversations too, side by side: one full of routing generations, one without them.

The whole integration is one `init()` call at module scope in `config.py`.

In [ ]:
view_file("config.py", show_path=True)

---

## 5. Kill the engineer, watch it resume

This task dies after its third live model call on the first attempt, right after the fine-tunes come back. Flyte retries it in a fresh container: the model turns it already paid for replay from their records, the evals and fine-tunes are cache hits, and the decision is produced once. In the pod logs, attempt 0 prints three `live model call` lines and the crash; attempt 1 prints four. In Grafana both attempts are one trace, with the replayed steps marked.

In [ ]:
from step5_crash_resume import resilient_engineer
run(resilient_engineer)

---

## 6. Bake off the engineer's brain (optional)

Same request, several models driving the engineer, in parallel, scored on whether they promoted something that meets it and how many runs they spent. Each agent model is its own agent version in Grafana. Skip if short on time.

In [ ]:
# from step6_bakeoff import bakeoff
# run(bakeoff, models=["anthropic:claude-opus-5", "anthropic:claude-haiku-4-5"])

---

## 7. Drift: a new kind of ticket

The support team adds a ninth queue, `data_request`, for GDPR-style tickets. Your router has never seen the label. See it first: the support agent on v2 tickets, still routing with the model you deployed in step 2. The GDPR tickets pile up in `other`, and the report flags them as belonging to a queue the router does not know. If you have Grafana, the router's own metrics show the same thing: the share of `other` climbs and confidence drops.

In [ ]:
drift = run(handle_tickets, n=36, router="oss", dataset="v2")
show(drift, ["routing_accuracy", "tickets_with_unknown_queue", "queue_distribution"])

---

## 8. The loop closes itself

Nobody should have to notice that. Deploy two triggers once: whenever a new version of your `support-tickets-<tag>` artifact lands, a run of `adapt` starts that you did not start. It is the same engineer, told "new tickets landed, check production first". It measures the live model on v2, finds it below the bar, retrains on v2, promotes and tests the deployment; that promotion fires the second trigger, which validates the new version on a T4. If production had still met the bar, `adapt` would have kept it and stopped.

In [ ]:
!flyte deploy step8_adaptive_loop.py observed_env
!flyte deploy validate_on_promote.py validate_env

Now publish the v2 tickets. Then open the runs list in the Flyte UI and watch: a run of `adapt` appears on its own (a minute or so), then a run of `validate_router`. Open the `adapt` run: it is a full decision page, made without you.

In [ ]:
from step8_adaptive_loop import publish_tickets
run(publish_tickets, version="v2")
print("watch the runs list: 'adapt' appears on its own, then 'validate_router'")

When the `adapt` run has finished, the support agent is fine again. Your `ticket-router-<tag>` artifact has a new version; open it in the UI and look at Versions and Lineage: from the app, back through the promotion, to the fine-tune, to the tickets that caused it.

In [ ]:
after_fix = run(handle_tickets, n=36, router="oss", dataset="v2", labels_from="v2")
show(after_fix, ["routing_accuracy", "tickets_with_unknown_queue", "queue_distribution"])

### The same fix, by hand (optional)

If you would rather watch the engineer do it in front of you, this is the same request without the trigger: tell the engineer what changed and let it work.

In [ ]:
# from step7_drift import day_two
# fixed = run(day_two)
# show(fixed["decision"], ["action", "promoted_model", "accuracy", "deployment_test_passed", "runs_spent"])

---

## Where to go from here

- Change the request: `run(engineer, min_accuracy=0.98, max_latency_ms=50)` is a different problem.
- Add a candidate to `factory.CANDIDATES`. Anything on the Hub with a chat template, or any encoder, works.
- Replace `tickets.py` with your own data. The tools do not care where the tickets come from.
- Turn on the approval gate: `os.environ["FACTORY_APPROVAL"] = "1"` and `promote` pauses in the Flyte UI until someone says yes.
- The README has the full write-up, the timings, and the gotchas we hit building this.